# Liquefied petroleum gas (LPG) - Colombia
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

In [ ]:
# librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model
from scipy.stats import norm

# Datos
import yfinance as yfin

In [ ]:
# FRED
from fredapi import Fred

In [ ]:
# FRED_API_KEY
fred = Fred(api_key='a3d8a45391e92cc5c30c9c255819c138')

In [ ]:
# Descargamos datos FRED para Propane Prices: Mont Belvieu, Texas
df_c = pd.DataFrame(fred.get_series('DPROPANEMBTX','2010-01-01'))
df_c = df_c.dropna()
df_c.index.names = ['Date']
df_c.columns = ["LPG"]
df_c

In [ ]:
# Descargamos datos FRED para la gasolina regular
raw_df_trm = yfin.download('COP=X', start='2010-01-01', multi_level_index=False)
# Select the 'Close' column, drop NaNs, and convert to a DataFrame with a new column name.
# This makes sure we are operating on a single column from the start.
df_trm = pd.DataFrame(raw_df_trm['Close'].dropna())
df_trm.columns = ["TRM"]
df_trm.index.names = ['Date'] # Ensure index name is set after modifications

# Remove outliers where TRM is less than 1000
df_trm = df_trm[df_trm['TRM'] >= 1600]
df_trm

In [ ]:
# Merge the DataFrames on the index
df = pd.merge(df_c, df_trm, on='Date')
df

In [ ]:
kg_per_gal = 2.04
costo_gal = 180/1000

df['df'] = (costo_gal + (1/kg_per_gal)*np.array(df['LPG']).astype(float))*np.array(df['TRM']).astype(float)
df

In [ ]:

# Calcular los retornos logarítmicos para la columna 'PPEt'
df['Rt_1'] = np.log(df['df'] / df['df'].shift(1))

print("Columna 'Rt' calculada exitosamente. Aquí están las primeras 5 filas con la nueva columna:")
display(df.head())

In [ ]:

# Calcular los retornos logarítmicos para la columna 'PPEt'
df['Rt_TRM'] = np.log(df['TRM'] / df['TRM'].shift(1))

print("Columna 'Rt' calculada exitosamente. Aquí están las primeras 5 filas con la nueva columna:")
display(df.head())

In [ ]:
import statsmodels.api as sm

# Realizar la prueba de Engle (ARCH test) sobre los retornos logarítmicos
# Es importante eliminar los valores NaN antes de la prueba
arch_test_results = sm.stats.diagnostic.het_arch(df['Rt_1'].dropna())

# Los resultados suelen venir en una tupla: (lm_statistic, p_value, f_statistic, f_p_value)
lm_statistic = arch_test_results[0]
p_value = arch_test_results[1]

print(f"Estadístico LM de la prueba ARCH: {lm_statistic:.4f}")
print(f"P-valor de la prueba ARCH: {p_value:.4f}")

if p_value < 0.05:
    print("\nConclusión: Se rechaza la hipótesis nula. Hay evidencia de efectos ARCH (heterocedasticidad condicional) en los retornos logarítmicos.")
else:
    print("\nConclusión: No se rechaza la hipótesis nula. No hay evidencia suficiente de efectos ARCH en los retornos logarítmicos.")

In [ ]:
from arch import arch_model

# Eliminar los valores NaN de los retornos logarítmicos
returns = df['Rt_1'].dropna()

# Crear y ajustar el modelo GARCH(1,1)
# mean='Constant' para el modelo de media constante
# vol='Garch' para el modelo de varianza GARCH
# p=1, q=1 para GARCH(1,1)
garch_model = arch_model(returns, mean='Zero', vol='GARCH', p=1, q=1)
garch_results = garch_model.fit(disp='off') # disp='off' para no mostrar la salida de cada iteración

print("Resumen del modelo GARCH(1,1):")
print(garch_results.summary())

In [ ]:
alpha_1 = garch_results.params['alpha[1]']
beta_1 = garch_results.params['beta[1]']

print(f"Alpha[1] coefficient: {alpha_1:.4f}")
print(f"Beta[1] coefficient: {beta_1:.4f}")

In [ ]:
# Obtener la volatilidad condicional (desviación estándar condicional) del modelo GARCH
conditional_volatility = garch_results.conditional_volatility

# Asumimos 252 días de negociación en un año para anualizar la volatilidad
volatilidad_anualizada_garch = conditional_volatility * np.sqrt(252)

print("Primeros 5 valores de la volatilidad anualizada GARCH:")
display(volatilidad_anualizada_garch.head())

print("\nÚltimo valor de la volatilidad anualizada GARCH:")
print(volatilidad_anualizada_garch.iloc[-1])

In [ ]:
from scipy import stats
from scipy.stats import norm, chi2, jarque_bera

# Obtener los residuales estandarizados del modelo GARCH
# Los residuales estandarizados son los errores divididos por la volatilidad condicional
garch_residuals = garch_results.std_resid

# Eliminar los valores NaN si los hubiera antes de la prueba
# Aunque en los residuales estandarizados del GARCH normalmente no hay NaN, es buena práctica
jb_value_garch_res, p_value_garch_res = stats.jarque_bera(garch_residuals.dropna())

print("Jarque-Bera value (Residuales GARCH): ", jb_value_garch_res)
print("p-value (Residuales GARCH): ", p_value_garch_res)

if p_value_garch_res > 0.05:
    print("\nConclusión: No se rechaza la hipótesis nula. Los residuales estandarizados del GARCH parecen seguir una distribución Normal.")
else:
    print("\nConclusión: Se rechaza la hipótesis nula. Los residuales estandarizados del GARCH no parecen seguir una distribución Normal.")

In [ ]:

# Obtener los residuales estandarizados al cuadrado
# Es importante eliminar los valores NaN si los hubiera
squared_garch_residuals = garch_results.std_resid.dropna()**2

# Realizar la prueba de Ljung-Box en los residuales estandarizados al cuadrado
# Un número común de lags para probar es 10 o 20
# qs es el estadístico de la prueba Q, pvalues son los p-valores para cada lag

ljung_box_results = sm.stats.diagnostic.acorr_ljungbox(squared_garch_residuals, lags=[10, 20], return_df=True)

print("Resultados de la prueba de Ljung-Box en los residuales estandarizados al cuadrado:")
display(ljung_box_results)

print("\nComentarios sobre la prueba de Ljung-Box:")
for index, row in ljung_box_results.iterrows():
    if row['lb_pvalue'] < 0.05:
      print(f"Para el lag {index}, se rechaza la hipótesis nula (p-value = {row['lb_pvalue']:.4f}).\n"
          "Esto indica que aún existe autocorrelación significativa en los residuales al cuadrado.\n"
          "Esto sugiere que el modelo GARCH podría no haber capturado completamente la dinámica de la volatilidad.\n")
    else:
      print(f"Para el lag {index}, no se rechaza la hipótesis nula (p-value = {row['lb_pvalue']:.4f}).\n"
          "No hay evidencia de autocorrelación significativa en los residuales al cuadrado.\n"
          "Esto sugiere que el modelo GARCH ha capturado adecuadamente la dinámica de la volatilidad.\n")

In [ ]:
import statsmodels.api as sm

# Realizar la prueba de Engle (ARCH test) sobre los retornos logarítmicos
# Es importante eliminar los valores NaN antes de la prueba
arch_test_results = sm.stats.diagnostic.het_arch(df['Rt_TRM'].dropna())

# Los resultados suelen venir en una tupla: (lm_statistic, p_value, f_statistic, f_p_value)
lm_statistic = arch_test_results[0]
p_value = arch_test_results[1]

print(f"Estadístico LM de la prueba ARCH: {lm_statistic:.4f}")
print(f"P-valor de la prueba ARCH: {p_value:.4f}")

if p_value < 0.05:
    print("\nConclusión: Se rechaza la hipótesis nula. Hay evidencia de efectos ARCH (heterocedasticidad condicional) en los retornos logarítmicos.")
else:
    print("\nConclusión: No se rechaza la hipótesis nula. No hay evidencia suficiente de efectos ARCH en los retornos logarítmicos.")

In [ ]:
from arch import arch_model

# Eliminar los valores NaN de los retornos logarítmicos
returns_TRM = df['Rt_TRM'].dropna()

# Crear y ajustar el modelo GARCH(1,1)
# mean='Constant' para el modelo de media constante
# vol='Garch' para el modelo de varianza GARCH
# p=1, q=1 para GARCH(1,1)
garch_model_TRM = arch_model(returns_TRM, mean='Zero', vol='GARCH', p=1, q=1)
garch_results_TRM = garch_model_TRM.fit(disp='off') # disp='off' para no mostrar la salida de cada iteración

print("Resumen del modelo GARCH(1,1):")
print(garch_results_TRM.summary())

In [ ]:
alpha_1_TRM = garch_results_TRM.params['alpha[1]']
beta_1_TRM = garch_results_TRM.params['beta[1]']

print(f"Alpha[1] coefficient: {alpha_1_TRM:.4f}")
print(f"Beta[1] coefficient: {beta_1_TRM:.4f}")

In [ ]:
# Obtener la volatilidad condicional (desviación estándar condicional) del modelo GARCH
conditional_volatility_TRM = garch_results_TRM.conditional_volatility

# Asumimos 252 días de negociación en un año para anualizar la volatilidad
volatilidad_anualizada_garch_TRM = conditional_volatility_TRM * np.sqrt(252)

print("Primeros 5 valores de la volatilidad anualizada GARCH:")
display(volatilidad_anualizada_garch_TRM.head())

print("\nÚltimo valor de la volatilidad anualizada GARCH:")
print(volatilidad_anualizada_garch_TRM.iloc[-1])

In [ ]:
from scipy import stats
from scipy.stats import norm, chi2, jarque_bera

# Obtener los residuales estandarizados del modelo GARCH
# Los residuales estandarizados son los errores divididos por la volatilidad condicional
garch_residuals_TRM = garch_results_TRM.std_resid

# Eliminar los valores NaN si los hubiera antes de la prueba
# Aunque en los residuales estandarizados del GARCH normalmente no hay NaN, es buena práctica
jb_value_garch_res_TRM, p_value_garch_res_TRM = stats.jarque_bera(garch_residuals_TRM.dropna())

print("Jarque-Bera value (Residuales GARCH): ", jb_value_garch_res_TRM)
print("p-value (Residuales GARCH): ", p_value_garch_res_TRM)

if p_value_garch_res_TRM > 0.05:
    print("\nConclusión: No se rechaza la hipótesis nula. Los residuales estandarizados del GARCH parecen seguir una distribución Normal.")
else:
    print("\nConclusión: Se rechaza la hipótesis nula. Los residuales estandarizados del GARCH no parecen seguir una distribución Normal.")

In [ ]:

# Obtener los residuales estandarizados al cuadrado
# Es importante eliminar los valores NaN si los hubiera
squared_garch_residuals_TRM = garch_results_TRM.std_resid.dropna()**2

# Realizar la prueba de Ljung-Box en los residuales estandarizados al cuadrado
# Un número común de lags para probar es 10 o 20
# qs es el estadístico de la prueba Q, pvalues son los p-valores para cada lag

ljung_box_results_TRM = sm.stats.diagnostic.acorr_ljungbox(squared_garch_residuals_TRM, lags=[10, 20], return_df=True)

print("Resultados de la prueba de Ljung-Box en los residuales estandarizados al cuadrado:")
display(ljung_box_results_TRM)

print("\nComentarios sobre la prueba de Ljung-Box:")
for index, row in ljung_box_results_TRM.iterrows():
    if row['lb_pvalue'] < 0.05:
      print(f"Para el lag {index}, se rechaza la hipótesis nula (p-value = {row['lb_pvalue']:.4f}).\n"
          "Esto indica que aún existe autocorrelación significativa en los residuales al cuadrado.\n"
          "Esto sugiere que el modelo GARCH podría no haber capturado completamente la dinámica de la volatilidad.\n")
    else:
      print(f"Para el lag {index}, no se rechaza la hipótesis nula (p-value = {row['lb_pvalue']:.4f}).\n"
          "No hay evidencia de autocorrelación significativa en los residuales al cuadrado.\n"
          "Esto sugiere que el modelo GARCH ha capturado adecuadamente la dinámica de la volatilidad.\n")

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Obtener los residuales estandarizados al cuadrado
squared_garch_residuals = garch_results.std_resid.dropna()**2

# Crear el autocorrelograma (ACF)
fig, ax = plt.subplots(figsize=(12, 6))
sm.graphics.tsa.plot_acf(squared_garch_residuals, lags=20, ax=ax)
ax.set_title('Autocorrelograma (ACF) de los Residuales Estandarizados al Cuadrado del precio del propano')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Obtener los residuales estandarizados al cuadrado
squared_garch_residuals = garch_results.std_resid.dropna()**2

# Crear el autocorrelograma parcial (PACF)
fig, ax = plt.subplots(figsize=(12, 6))
sm.graphics.tsa.plot_pacf(squared_garch_residuals, lags=20, ax=ax)
ax.set_title('Autocorrelograma Parcial (PACF) de los Residuales Estandarizados al Cuadrado del precio del propano')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Obtener los residuales estandarizados al cuadrado
squared_garch_residuals_TRM = garch_results_TRM.std_resid.dropna()**2

# Crear el autocorrelograma (ACF)
fig, ax = plt.subplots(figsize=(12, 6))
sm.graphics.tsa.plot_acf(squared_garch_residuals_TRM, lags=20, ax=ax)
ax.set_title('Autocorrelograma (ACF) de los Residuales Estandarizados al Cuadrado del TRM')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Obtener los residuales estandarizados al cuadrado
squared_garch_residuals_TRM = garch_results_TRM.std_resid.dropna()**2

# Crear el autocorrelograma parcial (PACF)
fig, ax = plt.subplots(figsize=(10, 5))
sm.graphics.tsa.plot_pacf(squared_garch_residuals_TRM, lags=20, ax=ax)
ax.set_title('Autocorrelograma Parcial (PACF) de los Residuales Estandarizados al Cuadrado del TRM')
plt.show()

In [ ]:
#PUNTO 3


# Obtener el último precio spot de la columna 'df'
last_spot_price = df['df'].iloc[-1]
last_spot_date = df.index[-1]

# Definir el costo de almacenamiento (anual)
storage_cost_annual = 0.01  # 1%

# Definir la fecha objetivo para la cual se calculará el futuro
target_date = pd.to_datetime('2026-12-31')

# Calcular el tiempo hasta la madurez en años
time_to_maturity = (target_date - last_spot_date).days / 365.25

# Asumir una tasa libre de riesgo anual (por ejemplo, 5%).
# Es importante señalar que esta es una suposición y debería ser reemplazada por una tasa adecuada al contexto.
risk_free_rate_annual = 0.1025  # 5%

# Calcular el valor futuro utilizando la fórmula: F = S * e^((r + u) * T)
# donde S es el precio spot, r es la tasa libre de riesgo, u es el costo de almacenamiento, y T es el tiempo.
future_price_propane = last_spot_price * np.exp((risk_free_rate_annual + storage_cost_annual) * time_to_maturity)

print(f"Último precio spot conocido (columna 'df'): {last_spot_price:.4f} en la fecha {last_spot_date.strftime('%Y-%m-%d')}")
print(f"Costo de almacenamiento anual: {storage_cost_annual * 100}%")
print(f"Tasa libre de riesgo anual asumida: {risk_free_rate_annual * 100}%")
print(f"Tiempo hasta la madurez (en años): {time_to_maturity:.2f}")
print(f"El valor futuro estimado del propano para diciembre de 2026 (PPi) es: {future_price_propane:.4f}")
print("")
print("")
print("El valor futuro estimado del propano Mont Belvieu para diciembre de 2026 es 2217.57, calculado a partir de un precio spot de 2023.71, una tasa libre de riesgo de 10.25% y un costo de almacenamiento de 1%. El hecho de que el precio futuro sea mayor que el precio spot es consistente con el modelo de cost of carry, ya que el precio incorpora los costos financieros y de almacenamiento asociados a mantener el commodity hasta la fecha de entrega. Esto implica que el precio esperado del propano aumentaría aproximadamente 9.6% hacia diciembre de 2026. Para las industrias que utilizan GLP importado, este resultado sugiere un posible incremento en sus costos de abastecimiento, lo que resalta la importancia de implementar estrategias de cobertura para mitigar el riesgo de aumentos en el precio del combustible.")